# Task A Task-B Style Stacking Classifier

This notebook overwrites the previous Task A FC neural-network experiment with a classification experiment that mirrors the Task B ensemble design:

- `XGBClassifier`
- `RandomForestClassifier`
- `HistGradientBoostingClassifier`
- `LogisticRegression`
- `MLPClassifier`
- optional native-categorical `CatBoostClassifier`
- `LogisticRegression` meta-learner inside `StackingClassifier`

The notebook:
- reuses the Task A preprocessing family for one-hot, scaled, and native-categorical branches
- runs 5-fold stratified cross-validation for the new experiment
- compares the new stack against the stored FC NN and tree + CatBoost baselines
- writes an executed markdown report to `task_a/reports/taskA_taskb_style_stack_vs_baselines.md`


In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
OUTER_FOLDS = 5
STACK_FOLDS = 2
MODEL_THREADS = 2
INCLUDE_CATBOOST_BASE = False
RUN_FULL_TRAIN_PREDICTION = True

DATA_DIR = Path("data/creditsense-ai1215")
TRAIN_PATH = DATA_DIR / "credit_train.csv"
TEST_PATH = DATA_DIR / "credit_test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

BASELINE_RESULTS_PATH = Path("task_a/artifacts/taskA_tree_catboost_cv_results.json")
REPORT_PATH = Path("task_a/reports/taskA_taskb_style_stack_vs_baselines.md")
PREDICTION_PATH = Path("submissions/submission.csv")

CLASS_LABELS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["VeryLow(0)", "Low(1)", "Moderate(2)", "High(3)", "VeryHigh(4)"]

print("Configuration loaded.")
print(f"Outer folds: {OUTER_FOLDS}")
print(f"Stacking inner folds: {STACK_FOLDS}")
print(f"Include CatBoost base learner: {INCLUDE_CATBOOST_BASE}")
print(f"Model threads per tree learner: {MODEL_THREADS}")


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
baseline_payload = json.loads(BASELINE_RESULTS_PATH.read_text(encoding="utf-8"))

X = train_df.drop(columns=["RiskTier", "InterestRate"])
y = train_df["RiskTier"].astype(int)

baseline_summary_df = pd.concat(
    [
        pd.DataFrame(baseline_payload["summary"]),
        pd.DataFrame(baseline_payload["nn_summary"]),
    ],
    ignore_index=True,
)

print(f"Train rows/cols: {train_df.shape}")
print(f"Test rows/cols: {test_df.shape}")
print(f"Task A feature matrix: {X.shape}")
print("RiskTier distribution:")
print(y.value_counts().sort_index().to_string())
print("\nStored baselines:")
print(baseline_summary_df.round(4).to_string(index=False))


In [ ]:
TASK_A_ZERO_FILL_COLS = [
    "StudentLoanOutstandingBalance",
    "MortgageOutstandingBalance",
    "PropertyValue",
    "InvestmentPortfolioValue",
    "VehicleValue",
    "AutoLoanOutstandingBalance",
    "SecondaryMonthlyIncome",
    "CollateralValue",
]

TASK_A_MONEY_CLIP_COLS = [
    "AnnualIncome",
    "MonthlyGrossIncome",
    "SecondaryMonthlyIncome",
    "TotalMonthlyIncome",
    "SavingsBalance",
    "CheckingBalance",
    "InvestmentPortfolioValue",
    "PropertyValue",
    "VehicleValue",
    "TotalAssets",
    "MortgageOutstandingBalance",
    "AutoLoanOutstandingBalance",
    "StudentLoanOutstandingBalance",
    "TotalCreditLimit",
    "RequestedLoanAmount",
    "CollateralValue",
    "MonthlyPaymentEstimate",
]

TASK_A_RATIO_CLIP_COLS = ["LoanToIncomeRatio", "PaymentToIncomeRatio"]
TASK_A_LOG_COLS = TASK_A_MONEY_CLIP_COLS.copy()


def _fit_tabular_preprocessor(
    X_train_raw: pd.DataFrame,
    *,
    zero_fill_cols: list[str],
    clip_cols: list[str],
    log_cols: list[str],
) -> dict[str, Any]:
    X_train_raw = X_train_raw.copy()

    base_cols = X_train_raw.columns.tolist()
    num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
    missing_flag_cols = [col for col in base_cols if X_train_raw[col].isna().any()]

    zero_fill_cols = [col for col in zero_fill_cols if col in num_cols]
    median_fill_map = {
        col: float(X_train_raw[col].median())
        for col in num_cols
        if col not in zero_fill_cols
    }

    clip_map: dict[str, float] = {}
    for col in clip_cols:
        if col not in num_cols:
            continue
        non_null = X_train_raw[col].dropna()
        clip_map[col] = float(non_null.quantile(0.99)) if not non_null.empty else 0.0

    prep = {
        "base_cols": base_cols,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "missing_flag_cols": missing_flag_cols,
        "zero_fill_cols": zero_fill_cols,
        "median_fill_map": median_fill_map,
        "clip_map": clip_map,
        "log_cols": [col for col in log_cols if col in num_cols],
    }

    transformed_train = _transform_tabular(X_train_raw, prep, align=False)
    prep["feature_columns"] = transformed_train.columns.tolist()
    return prep


def _transform_tabular(X_raw: pd.DataFrame, prep: dict[str, Any], align: bool = True) -> pd.DataFrame:
    working = X_raw.copy()

    for col in prep["missing_flag_cols"]:
        working[f"{col}_is_missing"] = working[col].isna().astype(np.int8)

    for col in prep["cat_cols"]:
        working[col] = working[col].fillna("Missing").astype(str)

    for col in prep["zero_fill_cols"]:
        working[col] = working[col].fillna(0.0)

    for col, med in prep["median_fill_map"].items():
        working[col] = working[col].fillna(med)

    for col, upper in prep["clip_map"].items():
        working[col] = working[col].clip(upper=upper)

    for col in prep["log_cols"]:
        working[col] = np.log1p(working[col])

    numeric_frame = working[prep["num_cols"]].apply(pd.to_numeric)
    missing_cols = [f"{col}_is_missing" for col in prep["missing_flag_cols"]]
    missing_frame = (
        working[missing_cols].astype(np.int8)
        if missing_cols
        else pd.DataFrame(index=working.index)
    )
    cat_frame = pd.get_dummies(
        working[prep["cat_cols"]],
        prefix=prep["cat_cols"],
        prefix_sep="__",
        drop_first=False,
        dtype=np.int8,
    )

    transformed = pd.concat([numeric_frame, missing_frame, cat_frame], axis=1).fillna(0)
    if align and "feature_columns" in prep:
        transformed = transformed.reindex(columns=prep["feature_columns"], fill_value=0)
    return transformed


def fit_task_a_preprocessor(X_train_raw: pd.DataFrame) -> dict[str, Any]:
    return _fit_tabular_preprocessor(
        X_train_raw,
        zero_fill_cols=TASK_A_ZERO_FILL_COLS,
        clip_cols=TASK_A_MONEY_CLIP_COLS + TASK_A_RATIO_CLIP_COLS,
        log_cols=TASK_A_LOG_COLS,
    )


def transform_task_a(X_raw: pd.DataFrame, prep: dict[str, Any]) -> pd.DataFrame:
    transformed = _transform_tabular(X_raw, prep, align=True)
    return transformed.reindex(columns=prep["feature_columns"], fill_value=0)


def fit_task_a_catboost_preprocessor(X_train_raw: pd.DataFrame) -> dict[str, Any]:
    X_train_raw = X_train_raw.copy()

    base_cols = X_train_raw.columns.tolist()
    num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
    missing_flag_cols = [col for col in base_cols if X_train_raw[col].isna().any()]

    zero_fill_cols = [col for col in TASK_A_ZERO_FILL_COLS if col in num_cols]
    clip_cols = [col for col in TASK_A_MONEY_CLIP_COLS + TASK_A_RATIO_CLIP_COLS if col in num_cols]
    log_cols = [col for col in TASK_A_LOG_COLS if col in num_cols]

    clip_map: dict[str, float] = {}
    for col in clip_cols:
        non_null = X_train_raw[col].dropna()
        clip_map[col] = float(non_null.quantile(0.99)) if not non_null.empty else 0.0

    prep = {
        "base_cols": base_cols,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "missing_flag_cols": missing_flag_cols,
        "zero_fill_cols": zero_fill_cols,
        "clip_map": clip_map,
        "log_cols": log_cols,
    }

    transformed_train = transform_task_a_catboost(X_train_raw, prep)
    prep["feature_columns"] = transformed_train.columns.tolist()
    return prep


def transform_task_a_catboost(X_raw: pd.DataFrame, prep: dict[str, Any]) -> pd.DataFrame:
    working = X_raw.copy()

    for col in prep["missing_flag_cols"]:
        working[f"{col}_is_missing"] = working[col].isna().astype(np.int8)

    for col in prep["cat_cols"]:
        working[col] = working[col].fillna("Missing").astype(str)

    for col in prep["zero_fill_cols"]:
        working[col] = working[col].fillna(0.0)

    for col, upper in prep["clip_map"].items():
        working[col] = working[col].clip(upper=upper)

    for col in prep["log_cols"]:
        working[col] = np.log1p(working[col])

    for col in prep["num_cols"]:
        working[col] = pd.to_numeric(working[col], errors="coerce")

    missing_cols = [f"{col}_is_missing" for col in prep["missing_flag_cols"]]
    ordered_cols = prep["base_cols"] + missing_cols
    transformed = working.reindex(columns=ordered_cols)
    if "feature_columns" in prep:
        transformed = transformed.reindex(columns=prep["feature_columns"], fill_value=0)
    return transformed


In [ ]:
class TaskATabularTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, scale_numeric: bool = False):
        self.scale_numeric = scale_numeric

    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "TaskATabularTransformer":
        self.prep_ = fit_task_a_preprocessor(X)
        transformed = transform_task_a(X, self.prep_)
        self.feature_columns_ = transformed.columns.tolist()
        self.continuous_cols_ = [col for col in self.prep_["num_cols"] if col in transformed.columns]
        self.feature_count_ = transformed.shape[1]
        if self.scale_numeric:
            self.scaler_ = StandardScaler()
            self.scaler_.fit(transformed[self.continuous_cols_])
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        transformed = transform_task_a(X, self.prep_).copy()
        if self.scale_numeric:
            transformed.loc[:, self.continuous_cols_] = self.scaler_.transform(
                transformed[self.continuous_cols_]
            )
        return transformed


class TaskACatBoostNativeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        iterations: int = 150,
        learning_rate: float = 0.05,
        depth: int = 6,
        random_state: int = RANDOM_STATE,
        thread_count: int = MODEL_THREADS,
    ):
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.random_state = random_state
        self.thread_count = thread_count

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "TaskACatBoostNativeClassifier":
        self.prep_ = fit_task_a_catboost_preprocessor(X)
        X_fit = transform_task_a_catboost(X, self.prep_)
        self.model_ = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=self.iterations,
            learning_rate=self.learning_rate,
            depth=self.depth,
            random_seed=self.random_state,
            thread_count=self.thread_count,
            verbose=False,
            allow_writing_files=False,
        )
        self.model_.fit(X_fit, y, cat_features=self.prep_["cat_cols"])
        self.classes_ = np.asarray(sorted(pd.Series(y).unique()))
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        X_infer = transform_task_a_catboost(X, self.prep_)
        return self.model_.predict_proba(X_infer)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        X_infer = transform_task_a_catboost(X, self.prep_)
        pred = self.model_.predict(X_infer)
        return np.asarray(pred).reshape(-1).astype(int)


def build_task_b_style_estimators(include_catboost: bool = INCLUDE_CATBOOST_BASE) -> list[tuple[str, Any]]:
    estimators: list[tuple[str, Any]] = [
        (
            "xgb",
            Pipeline(
                steps=[
                    ("preprocess", TaskATabularTransformer(scale_numeric=False)),
                    (
                        "model",
                        XGBClassifier(
                            n_estimators = 1035,
                            max_depth = 6,
                            learning_rate = 0.03992306657672681,
                            subsample=0.8524337374155861,
                            colsample_bytree=0.8837262828873732,
                            objective="multi:softprob",
                            num_class=len(CLASS_LABELS),
                            eval_metric="mlogloss",
                            tree_method="hist",
                            random_state=RANDOM_STATE,
                            n_jobs=MODEL_THREADS,
                        ),
                    ),
                ]
            ),
        ),
        (
            "rf",
            Pipeline(
                steps=[
                    ("preprocess", TaskATabularTransformer(scale_numeric=False)),
                    (
                        "model",
                        RandomForestClassifier(
                            n_estimators=150,
                            max_depth=None,
                            random_state=RANDOM_STATE,
                            n_jobs=MODEL_THREADS,
                        ),
                    ),
                ]
            ),
        ),
        (
            "hgb",
            Pipeline(
                steps=[
                    ("preprocess", TaskATabularTransformer(scale_numeric=False)),
                    (
                        "model",
                        HistGradientBoostingClassifier(
                            max_iter=150,
                            learning_rate=0.05,
                            max_depth=6,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
        ),
        (
            "lr",
            Pipeline(
                steps=[
                    ("preprocess", TaskATabularTransformer(scale_numeric=True)),
                    (
                        "model",
                        LogisticRegression(
                            max_iter=2000,
                            multi_class="multinomial",
                            solver="lbfgs",
                        ),
                    ),
                ]
            ),
        ),
        (
            "mlp",
            Pipeline(
                steps=[
                    ("preprocess", TaskATabularTransformer(scale_numeric=True)),
                    (
                        "model",
                        MLPClassifier(
                            hidden_layer_sizes=(128, 64),
                            activation="relu",
                            solver="adam",
                            alpha=0.0001,
                            batch_size="auto",
                            learning_rate_init=0.001,
                            max_iter=150,
                            early_stopping=True,
                            n_iter_no_change=20,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
        ),
    ]

    if include_catboost:
        estimators.append(
            (
                "catboost",
                TaskACatBoostNativeClassifier(
                    iterations=150,
                    learning_rate=0.05,
                    depth=6,
                    random_state=RANDOM_STATE,
                    thread_count=MODEL_THREADS,
                ),
            )
        )
    return estimators


def build_task_b_style_stacker(include_catboost: bool = INCLUDE_CATBOOST_BASE) -> StackingClassifier:
    return StackingClassifier(
        estimators=build_task_b_style_estimators(include_catboost=include_catboost),
        final_estimator=LogisticRegression(
            max_iter=2000,
            multi_class="multinomial",
            solver="lbfgs",
        ),
        cv=STACK_FOLDS,
        stack_method="predict_proba",
        n_jobs=1,
        passthrough=False,
    )


In [ ]:
def dataframe_to_markdown(df: pd.DataFrame, index: bool = False, float_digits: int = 4) -> str:
    working = df.copy()
    if index:
        index_name = working.index.name or "index"
        working = working.reset_index().rename(columns={"index": index_name})

    def fmt(value: Any) -> str:
        if pd.isna(value):
            return ""
        if isinstance(value, (np.integer, int)):
            return str(int(value))
        if isinstance(value, (np.floating, float)):
            value = float(value)
            if abs(value - round(value)) < 1e-12:
                return str(int(round(value)))
            return f"{value:.{float_digits}f}"
        return str(value)

    header = "| " + " | ".join(map(str, working.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(working.columns)) + " |"
    rows = [
        "| " + " | ".join(fmt(value) for value in row) + " |"
        for row in working.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator] + rows)


def evaluate_model_cv(
    X_raw: pd.DataFrame,
    y_raw: pd.Series,
    *,
    include_catboost: bool = INCLUDE_CATBOOST_BASE,
) -> dict[str, Any]:
    skf = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_pred = np.full(len(X_raw), -1, dtype=int)
    fold_rows = []
    started = time.time()

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw), start=1):
        X_train = X_raw.iloc[train_idx].reset_index(drop=True)
        X_val = X_raw.iloc[val_idx].reset_index(drop=True)
        y_train = y_raw.iloc[train_idx].reset_index(drop=True)
        y_val = y_raw.iloc[val_idx].reset_index(drop=True)

        model = build_task_b_style_stacker(include_catboost=include_catboost)
        fold_started = time.time()
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val).astype(int)
        oof_pred[val_idx] = val_pred

        rf_pre = model.named_estimators_["rf"].named_steps["preprocess"]
        fold_accuracy = float(accuracy_score(y_val, val_pred))
        fold_macro_f1 = float(f1_score(y_val, val_pred, average="macro"))
        fold_rows.append(
            {
                "fold": fold_idx,
                "train_rows": len(train_idx),
                "val_rows": len(val_idx),
                "feature_count": int(rf_pre.feature_count_),
                "base_models": len(model.named_estimators_),
                "elapsed_minutes": round((time.time() - fold_started) / 60.0, 2),
                "accuracy": fold_accuracy,
                "macro_f1": fold_macro_f1,
            }
        )

        print(
            f"Fold {fold_idx}/{OUTER_FOLDS}: "
            f"accuracy={fold_accuracy:.4f}, "
            f"macro_f1={fold_macro_f1:.4f}, "
            f"feature_count={rf_pre.feature_count_}, "
            f"minutes={fold_rows[-1]['elapsed_minutes']:.2f}",
            flush=True,
        )

    if (oof_pred < 0).any():
        raise ValueError("OOF predictions did not cover every training row.")

    model_name = (
        "Task B-style StackingClassifier + CatBoost"
        if include_catboost
        else "Task B-style StackingClassifier"
    )
    fold_metrics_df = pd.DataFrame(fold_rows)
    summary_df = pd.DataFrame(
        [
            {
                "model": model_name,
                "accuracy_mean": float(fold_metrics_df["accuracy"].mean()),
                "accuracy_std": float(fold_metrics_df["accuracy"].std(ddof=1)),
                "macro_f1_mean": float(fold_metrics_df["macro_f1"].mean()),
                "macro_f1_std": float(fold_metrics_df["macro_f1"].std(ddof=1)),
                "elapsed_minutes": round((time.time() - started) / 60.0, 2),
            }
        ]
    )
    confusion_df = pd.DataFrame(
        confusion_matrix(y_raw.to_numpy(), oof_pred, labels=CLASS_LABELS),
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )
    report_df = (
        pd.DataFrame(
            classification_report(
                y_raw.to_numpy(),
                oof_pred,
                labels=CLASS_LABELS,
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )
        )
        .T.reset_index()
        .rename(columns={"index": "label"})
    )

    return {
        "fold_metrics_df": fold_metrics_df,
        "summary_df": summary_df,
        "confusion_df": confusion_df,
        "report_df": report_df,
        "oof_pred": oof_pred,
    }


def build_comparison_table(experiment_summary_df: pd.DataFrame, baseline_summary_df: pd.DataFrame) -> pd.DataFrame:
    comparison_df = pd.concat([baseline_summary_df, experiment_summary_df], ignore_index=True, sort=False)
    comparison_df = comparison_df[
        ["model", "accuracy_mean", "accuracy_std", "macro_f1_mean", "macro_f1_std", "elapsed_minutes"]
    ].copy()

    tree_row = comparison_df.loc[comparison_df["model"] == "Tree stack + CatBoost"].iloc[0]
    nn_row = comparison_df.loc[comparison_df["model"] == "FC neural network"].iloc[0]
    comparison_df["delta_vs_tree_accuracy"] = comparison_df["accuracy_mean"] - float(tree_row["accuracy_mean"])
    comparison_df["delta_vs_tree_macro_f1"] = comparison_df["macro_f1_mean"] - float(tree_row["macro_f1_mean"])
    comparison_df["delta_vs_nn_accuracy"] = comparison_df["accuracy_mean"] - float(nn_row["accuracy_mean"])
    comparison_df["delta_vs_nn_macro_f1"] = comparison_df["macro_f1_mean"] - float(nn_row["macro_f1_mean"])
    return comparison_df


EXPERIMENT_RESULTS = evaluate_model_cv(X, y, include_catboost=INCLUDE_CATBOOST_BASE)
COMPARISON_DF = build_comparison_table(EXPERIMENT_RESULTS["summary_df"], baseline_summary_df)

print("\nNew experiment summary:")
print(EXPERIMENT_RESULTS["summary_df"].round(4).to_string(index=False))
print("\nComparison vs stored baselines:")
print(COMPARISON_DF.round(4).to_string(index=False))


In [ ]:

FULL_FIT_OUTPUT: dict[str, Any] | None = None

if RUN_FULL_TRAIN_PREDICTION:
    final_model = build_task_b_style_stacker(include_catboost=INCLUDE_CATBOOST_BASE)
    final_model.fit(X, y)
    test_pred = final_model.predict(test_df).astype(int)

    submission = sample_submission[["Id"]].copy()
    submission["RiskTier"] = test_pred

    interest_rate_source = "sample_submission.csv"
    if PREDICTION_PATH.exists():
        existing_submission = pd.read_csv(PREDICTION_PATH)
        if "InterestRate" in existing_submission.columns and len(existing_submission) == len(submission):
            submission["InterestRate"] = existing_submission["InterestRate"].to_numpy()
            interest_rate_source = str(PREDICTION_PATH)
    elif "InterestRate" in sample_submission.columns:
        submission["InterestRate"] = sample_submission["InterestRate"].to_numpy()

    ordered_cols = ["Id", "RiskTier"] + ([col for col in ["InterestRate"] if col in submission.columns])
    submission = submission[ordered_cols]
    submission.to_csv(PREDICTION_PATH, index=False)

    FULL_FIT_OUTPUT = {
        "prediction_path": str(PREDICTION_PATH),
        "rows": len(submission),
        "interest_rate_source": interest_rate_source,
        "distribution": (
            submission["RiskTier"]
            .value_counts()
            .sort_index()
            .rename_axis("RiskTier")
            .reset_index(name="count")
        ),
    }
    print(f"Saved submission to {PREDICTION_PATH}")
    print(f"InterestRate column source: {interest_rate_source}")
    print(FULL_FIT_OUTPUT["distribution"].to_string(index=False))
else:
    print("Full-train prediction disabled. Set RUN_FULL_TRAIN_PREDICTION=True to generate a submission artifact.")


In [ ]:
experiment_name = EXPERIMENT_RESULTS["summary_df"].loc[0, "model"]
new_summary = EXPERIMENT_RESULTS["summary_df"].iloc[0]

report_lines = [
    "# Task A Task-B Style Stacking Classifier",
    "",
    "## Objective",
    "Run a new Task A classification experiment that mirrors the Task B ensemble structure, optionally adds a native-categorical CatBoost base learner, and compares the 5-fold CV results against the stored FC NN and tree + CatBoost baselines.",
    "",
    "## Repo Layout Used",
    "- Raw data: `data/creditsense-ai1215/`",
    "- Task A scripts, reports, artifacts: `task_a/`",
    "- Task B notebook and regression artifact: `task_b/`",
    "- Competition submissions: `submissions/`",
    "",
    "## New Experiment",
    f"- Model: `{experiment_name}`",
    "- Base learners:",
    "  - `XGBClassifier`",
    "  - `RandomForestClassifier`",
    "  - `HistGradientBoostingClassifier`",
    "  - `LogisticRegression`",
    "  - `MLPClassifier`",
    f"  - `CatBoostClassifier` included: `{INCLUDE_CATBOOST_BASE}`",
    "- Meta-learner: `LogisticRegression(multi_class='multinomial')` inside `StackingClassifier`",
    f"- Outer CV folds: `{OUTER_FOLDS}`",
    f"- Inner stacking folds: `{STACK_FOLDS}`",
    "",
    "## Task A Preprocessing",
    "- One-hot branch for XGB, RF, HGB, logistic, and MLP uses the upgraded Task A preprocessing family.",
    "- Missing categorical values are filled with `Missing` and one-hot encoded.",
    "- Numeric missing indicators are added before imputation.",
    "- Structural Task A numeric columns are zero-filled where missingness semantically implies zero.",
    "- Remaining numeric columns are median-imputed with train-fold statistics only.",
    "- Money and ratio columns are clipped at the train-fold 99th percentile.",
    "- Money columns receive `log1p` after clipping.",
    "- Logistic and MLP branches additionally standardize only the processed numeric columns.",
    "- CatBoost keeps native categorical handling on its own Task A preprocessing branch.",
    "",
    "## Fold Metrics",
    dataframe_to_markdown(EXPERIMENT_RESULTS["fold_metrics_df"], index=False),
    "",
    "## Summary Comparison",
    dataframe_to_markdown(COMPARISON_DF, index=False),
    "",
    "## Confusion Matrix",
    dataframe_to_markdown(EXPERIMENT_RESULTS["confusion_df"], index=True, float_digits=0),
    "",
    "## Classification Report",
    dataframe_to_markdown(EXPERIMENT_RESULTS["report_df"], index=False),
    "",
    "## Artifacts",
    f"- Stored baseline summary source: `{BASELINE_RESULTS_PATH}`",
    f"- Executed report: `{REPORT_PATH}`",
]

if FULL_FIT_OUTPUT is None:
    report_lines.append("- Full-train submission generation was disabled for this run.")
else:
    report_lines.extend(
        [
            f"- Submission artifact: `{FULL_FIT_OUTPUT['prediction_path']}`",
            f"- Submission rows: `{FULL_FIT_OUTPUT['rows']}`",
        ]
    )

REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print(f"Saved markdown report to {REPORT_PATH}")
